# Portfolio Optimization — the Frontier via the Skill, + Shadow Prices (cuOpt QP)

The base `QP_portfolio_optimization` notebook **hand-codes** an efficient-frontier sweep (a manual loop over target returns). This sibling shows that following the `cuopt-multi-objective-exploration` skill **recreates that frontier as a named, systematic workflow** — anchor each objective → ε-constraint sweep (the return floor is the parametric bound) → filter dominated → read the frontier — with less ad-hoc scaffolding, and **adds the one thing the manual sweep omits**: the return-constraint **dual** (shadow price d(variance)/d(return)).

So the two examples are complementary tests of the skill: here it **reproduces** an existing frontier (return vs risk) with less manual work and surfaces the duals; the workforce MILP (`workforce_optimization/workforce_optimization_multiobjective.ipynb`) is the **net-new** case — and the deliberate contrast is that **a QP has constraint duals, an integer program does not.**

## Environment Setup

In [ ]:
import subprocess
import html
from IPython.display import display, HTML

def check_gpu():
    try:
        result = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=5)
        result.check_returncode()
        lines = result.stdout.splitlines()
        gpu_info = lines[2] if len(lines) > 2 else "GPU detected"
        gpu_info_escaped = html.escape(gpu_info)
        display(HTML(f"""
        <div style="border:2px solid #4CAF50;padding:10px;border-radius:10px;background:#e8f5e9;">
            <h3>✅ GPU is enabled</h3>
            <pre>{gpu_info_escaped}</pre>
        </div>
        """))
        return True
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired, FileNotFoundError, IndexError) as e:
        display(HTML("""
        <div style="border:2px solid red;padding:15px;border-radius:10px;background:#ffeeee;">
            <h3>⚠️ GPU not detected!</h3>
            <p>This notebook requires a <b>GPU runtime</b>.</p>

            <h4>If running in Google Colab:</h4>
            <ol>
              <li>Click on <b>Runtime → Change runtime type</b></li>
              <li>Set <b>Hardware accelerator</b> to <b>GPU</b></li>
              <li>Then click <b>Save</b> and <b>Runtime → Restart runtime</b>.</li>
            </ol>

            <h4>If running in Docker:</h4>
            <ol>
              <li>Ensure you have <b>NVIDIA Docker runtime</b> installed (<code>nvidia-docker2</code>)</li>
              <li>Run container with GPU support: <code>docker run --gpus all ...</code></li>
              <li>Or use: <code>docker run --runtime=nvidia ...</code> for older Docker versions</li>
              <li>Verify GPU access: <code>docker run --gpus all nvidia/cuda:12.0.0-base-ubuntu22.04 nvidia-smi</code></li>
            </ol>

            <p><b>Additional resources:</b></p>
            <ul>
              <li><a href="https://docs.nvidia.com/datacenter/cloud-native/container-toolkit/install-guide.html" target="_blank">NVIDIA Container Toolkit Installation Guide</a></li>
            </ul>
        </div>
        """))
        return False

check_gpu()

In [ ]:
# Uncomment for your CUDA version if cuOpt is not already installed (e.g., Google Colab):
# !pip install --upgrade --extra-index-url https://pypi.nvidia.com cuopt-cu12  # CUDA 12
# !pip install --upgrade --extra-index-url https://pypi.nvidia.com cuopt-cu13  # CUDA 13

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cuopt.linear_programming.problem import Problem, QuadraticExpression, MINIMIZE
print("Imports ready (cuOpt QP solver)")

## Data

Same simulated asset universe as `QP_portfolio_optimization` (annualized mean returns + covariance).

In [ ]:
# Simulate monthly returns with realistic assumptions
np.random.seed(7)

assets = ["Cash", "US Equity", "Intl Equity", "Bond", "REIT/Gold"]

annual_mean = np.array([0.02, 0.08, 0.075, 0.04, 0.06])
annual_vol = np.array([0.005, 0.16, 0.18, 0.06, 0.14])

corr = np.array([
    [1.00, 0.05, 0.05, 0.10, 0.05],
    [0.05, 1.00, 0.80, -0.10, 0.55],
    [0.05, 0.80, 1.00, -0.05, 0.50],
    [0.10, -0.10, -0.05, 1.00, 0.00],
    [0.05, 0.55, 0.50, 0.00, 1.00],
])

monthly_mean = annual_mean / 12.0
monthly_vol = annual_vol / np.sqrt(12.0)
monthly_cov = np.outer(monthly_vol, monthly_vol) * corr

n_months = 120
returns = np.random.multivariate_normal(monthly_mean, monthly_cov, size=n_months)

# Estimate annualized mean and covariance from the simulated data
mean_returns = returns.mean(axis=0) * 12.0
cov_matrix = np.cov(returns, rowvar=False) * 12.0

summary = pd.DataFrame(
    {
        "Annualized Return": mean_returns,
        "Annualized Volatility": np.sqrt(np.diag(cov_matrix)),
    },
    index=assets,
)

summary.style.format({"Annualized Return": "{:.2%}", "Annualized Volatility": "{:.2%}"})

## Min-variance QP with the return-constraint dual

This is the base notebook's `solve_min_variance_qp`, with one addition: we keep a handle on the `min_return` constraint and read its **`.DualValue`** after the solve. For the QP, that dual is the shadow price d(variance)/d(return).

In [ ]:
def solve_min_variance_qp_dual(cov_matrix, mean_returns, target_return=None, max_weight=None):
    n = len(mean_returns)
    prob = Problem("Portfolio_Optimization")
    ub = max_weight if max_weight is not None else 1.0
    w = [prob.addVariable(lb=0.0, ub=ub, name=f"w_{i}") for i in range(n)]

    quad = None
    for i in range(n):
        for j in range(n):
            c = float(cov_matrix[i, j])
            if abs(c) > 1e-12:
                term = c * w[i] * w[j]
                quad = term if quad is None else quad + term
    prob.setObjective(quad, sense=MINIMIZE)

    prob.addConstraint(sum(w) == 1, name="fully_invested")
    ret_con = None
    if target_return is not None:
        ret_expr = sum(float(mean_returns[i]) * w[i] for i in range(n))
        ret_con = prob.addConstraint(ret_expr >= float(target_return), name="min_return")

    prob.solve()
    status = prob.Status.name if hasattr(prob.Status, "name") else str(prob.Status)
    weights = np.array([w[i].Value for i in range(n)])
    port_ret = float(mean_returns @ weights)
    port_vol = float(np.sqrt(max(weights @ cov_matrix @ weights, 0.0)))
    dual = abs(float(ret_con.DualValue)) if ret_con is not None else 0.0   # shadow price d(var)/d(return)
    return {"weights": weights, "ret": port_ret, "vol": port_vol, "dual": dual, "status": status}

mv = solve_min_variance_qp_dual(cov_matrix, mean_returns)
print(f"Min-variance: status={mv['status']}, return={mv['ret']:.2%}, vol={mv['vol']:.2%}")

## Sweep the return target → frontier + shadow price

The ε-constraint sweep (return floor as the parametric bound), capturing the dual at each point. The skill's note applies: cuOpt's QP beta is PDLP (a first-order method), so the dual is accurate **to the solver's tolerance** — we keep points the solver reports as `Optimal` and flag any `PrimalFeasible`.

In [ ]:
min_ret = mv["ret"]
max_ret = float(mean_returns.max())
targets = np.linspace(min_ret, max_ret * 0.999, 25)

rets, vols, duals, flagged = [], [], [], 0
for t in targets:
    r = solve_min_variance_qp_dual(cov_matrix, mean_returns, target_return=t)
    if r["status"] not in ("Optimal", "PrimalFeasible"):
        continue
    if r["status"] != "Optimal":
        flagged += 1
    rets.append(r["ret"]); vols.append(r["vol"]); duals.append(r["dual"])

rets, vols, duals = map(np.array, (rets, vols, duals))
print(f"Frontier points: {len(rets)} | not certified-Optimal (PrimalFeasible): {flagged}")
print(f"Shadow price d(variance)/d(return): {duals.min():.3f} -> {duals.max():.3f} as required return rises")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(vols * 100, rets * 100, "o-", color="navy", lw=1.6)
axes[0].set_xlabel("Volatility (%)"); axes[0].set_ylabel("Expected Return (%)")
axes[0].set_title("Efficient frontier (return vs risk)"); axes[0].grid(alpha=0.3)

axes[1].plot(rets * 100, duals, "o-", color="purple", lw=1.6)
axes[1].set_xlabel("Required return (%)"); axes[1].set_ylabel("Shadow price  d(variance)/d(return)")
axes[1].set_title("Marginal risk cost of return (cuOpt QP dual)"); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Reading it

- The **frontier** (left) is the return-vs-risk Pareto set — every point is a min-variance portfolio for its return floor.
- The **dual** (right) is the *exchange rate* the skill asks you to report: how much variance you take on per extra unit of return. It rises along the frontier — the marginal cost of return gets steeper, which is exactly where a knee analysis pays off.

### Notes (honest)
- **Synthetic data** — the base notebook's simulated universe; demonstrates the method.
- **PDLP / first-order** — cuOpt's QP beta is a first-order solver, so the dual is optimal **to its convergence tolerance**, not exact arithmetic; points reported `PrimalFeasible` rather than `Optimal` are flagged above and could be tightened or dropped.
- **Continuous only** — these duals exist because the portfolio is a QP. The integer workforce model (`workforce_optimization_multiobjective.ipynb`) has **no constraint duals**; there you read the marginal cost off the frontier itself.

This adds the duals/interpretation step of the `cuopt-multi-objective-exploration` skill to cuOpt's existing portfolio frontier.

## License

SPDX-FileCopyrightText: Copyright (c) 2025 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: Apache-2.0

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.